# 22 — Wide-Timebin Classical Baselines vs Neural Models

This notebook compares the strongest neural models from Notebook 20 against stronger classical feature-engineering baselines.

Notebook 20 showed that ShallowConvNet was the strongest neural model overall, but it was compared mainly against raw-flat LDA/Logistic Regression.

This notebook adds:

- raw-flat LDA
- raw-flat Logistic Regression
- 50 ms timebin LDA
- 50 ms timebin Logistic Regression
- 50 ms wide-timebin LDA
- 50 ms wide-timebin Logistic Regression

Then it compares those classical baselines against the neural models already evaluated in Notebook 20:

- Current EEGNet
- Tuned EEGNet
- ShallowConvNet
- DeepConvNet
- DenseNet-style EEG CNN

The key question is:

> Does the best neural model outperform the best classical engineered-feature model?

Definitions:

- raw-flat: crop 0.0–0.6 s and flatten channels × time
- timebin: crop 0.0–0.6 s, average each channel in 50 ms bins
- wide-timebin: crop 0.0–0.8 s, average each channel in 50 ms bins

This notebook does not retrain neural models. It uses the saved Notebook 20 neural results.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment
from oddball_eeg.io import save_dataframe
from oddball_eeg.loader import load_one_subject_grouped
from oddball_eeg.metrics import classification_metric_dict
from oddball_eeg.splits import build_purged_chunk_splits
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)


warnings.filterwarnings("ignore")
set_pandas_display_options()

# ------------------------------------------------------------
# Input from Notebook 20
# ------------------------------------------------------------

NEURAL_TABLE_DIR = Path("results") / "thesis_artifacts" / "neural_model_diagnostics" / "tables"
NEURAL_SUBJECT_SUMMARY_PATH = NEURAL_TABLE_DIR / "subject_level_model_summary.csv"

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUT_ROOT = Path("results") / "thesis_artifacts" / "widetimebin_vs_neural_benchmark"
TABLE_DIR = OUT_ROOT / "tables"
FIG_DIR = OUT_ROOT / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Classical feature settings
# ------------------------------------------------------------

BIN_MS = 50

CLASSICAL_MODEL_NAMES = ["lda", "logreg"]

FEATURE_CONFIGS = {
    "raw_flat": {
        "label": "Raw-flat 0.0–0.6 s",
        "kind": "raw_flat",
        "crop_tmin": 0.0,
        "crop_tmax": 0.6,
    },
    "timebin_50ms": {
        "label": "Timebin 50 ms, 0.0–0.6 s",
        "kind": "timebin",
        "crop_tmin": 0.0,
        "crop_tmax": 0.6,
        "bin_ms": BIN_MS,
    },
    "wide_timebin_50ms": {
        "label": "Wide-timebin 50 ms, 0.0–0.8 s",
        "kind": "timebin",
        "crop_tmin": 0.0,
        "crop_tmax": 0.8,
        "bin_ms": BIN_MS,
    },
}

# Use the same fold design as the previous within-subject benchmark.
N_SPLITS_FALLBACK = 5
PURGE_NEIGHBORS_FALLBACK = 2

print("Neural subject summary:", NEURAL_SUBJECT_SUMMARY_PATH)
print("Output root:", OUT_ROOT)

In [ ]:
def normalize_subject_id(x) -> str:
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s


def dataset_label_from_cfg(cfg):
    if cfg.dataset.dataset == "ds006480":
        return f"{cfg.dataset.dataset}_{cfg.dataset.event_preset}"
    return cfg.dataset.dataset


def safe_get_experiment(exp_name: str):
    """
    Load experiment config. Keeps a small alias fallback in case local config
    uses n15 instead of max15, or vice versa.
    """
    aliases = {
        "ds003061_within_subject_max15": "ds003061_within_subject_n15",
        "ds005863_within_subject_max15": "ds005863_within_subject_n15",
        "ds006466_within_subject_max15": "ds006466_within_subject_n15",
        "ds006480_within_subject_max15": "ds006480_within_subject_n15",

        "ds003061_within_subject_n15": "ds003061_within_subject_max15",
        "ds005863_within_subject_n15": "ds005863_within_subject_max15",
        "ds006466_within_subject_n15": "ds006466_within_subject_max15",
        "ds006480_within_subject_n15": "ds006480_within_subject_max15",
    }

    try:
        return get_experiment(exp_name)
    except Exception:
        if exp_name in aliases:
            return get_experiment(aliases[exp_name])
        raise


def make_classical_model(model_name: str):
    model_name = str(model_name).lower()

    if model_name == "lda":
        return make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            LinearDiscriminantAnalysis(
                solver="lsqr",
                shrinkage="auto",
            ),
        )

    if model_name == "logreg":
        return make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            LogisticRegression(
                solver="lbfgs",
                max_iter=3000,
                class_weight="balanced",
            ),
        )

    raise ValueError(f"Unknown classical model: {model_name}")


def crop_epochs_to_window(X, times, tmin, tmax):
    times = np.asarray(times)
    mask = (times >= float(tmin)) & (times <= float(tmax))

    if mask.sum() < 2:
        raise ValueError(f"Crop window {tmin}–{tmax} produced too few time points.")

    return X[:, :, mask], times[mask]


def flatten_epochs(X):
    X = np.asarray(X, dtype=np.float32)
    return X.reshape(X.shape[0], -1)


def make_time_bins(times, bin_ms):
    times = np.asarray(times)
    sfreq = 1.0 / float(np.median(np.diff(times)))

    bin_samples = max(1, int(round((float(bin_ms) / 1000.0) * sfreq)))
    n_bins = len(times) // bin_samples
    keep = n_bins * bin_samples

    bins = []

    for b in range(n_bins):
        start = b * bin_samples
        stop = (b + 1) * bin_samples

        idx = np.arange(start, stop)

        bins.append({
            "bin_id": int(b),
            "start_idx": int(start),
            "stop_idx": int(stop),
            "time_start": float(times[start]),
            "time_stop": float(times[stop - 1]),
            "time_center": float(times[idx].mean()),
            "indices": idx,
        })

    return bins


def extract_timebin_features(X, times, bin_ms):
    """
    Convert epochs from trials × channels × time to trials × channels*bins.

    Each feature is the mean amplitude in one channel and one time bin.
    """
    bins = make_time_bins(times, bin_ms)

    if len(bins) == 0:
        raise ValueError("No time bins were created.")

    features = []

    for b in bins:
        idx = b["indices"]
        features.append(X[:, :, idx].mean(axis=2))

    return np.concatenate(features, axis=1), bins


def extract_features(X, times, feature_cfg):
    kind = feature_cfg["kind"]

    X_crop, times_crop = crop_epochs_to_window(
        X,
        times,
        feature_cfg["crop_tmin"],
        feature_cfg["crop_tmax"],
    )

    if kind == "raw_flat":
        X_feat = flatten_epochs(X_crop)
        meta = {
            "n_features": int(X_feat.shape[1]),
            "n_bins": np.nan,
            "time_start": float(times_crop[0]),
            "time_stop": float(times_crop[-1]),
        }
        return X_feat, meta

    if kind == "timebin":
        X_feat, bins = extract_timebin_features(
            X_crop,
            times_crop,
            feature_cfg["bin_ms"],
        )
        meta = {
            "n_features": int(X_feat.shape[1]),
            "n_bins": int(len(bins)),
            "time_start": float(times_crop[0]),
            "time_stop": float(times_crop[-1]),
        }
        return X_feat, meta

    raise ValueError(f"Unknown feature kind: {kind}")


def get_scores_and_predictions(clf, X_test):
    if hasattr(clf, "predict_proba"):
        scores = clf.predict_proba(X_test)[:, 1]
    elif hasattr(clf, "decision_function"):
        scores = clf.decision_function(X_test)
    else:
        scores = clf.predict(X_test)

    y_pred = clf.predict(X_test)
    return scores, y_pred

In [ ]:
if not NEURAL_SUBJECT_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Missing Notebook 20 subject summary:\n{NEURAL_SUBJECT_SUMMARY_PATH}"
    )

neural_subject_df = pd.read_csv(NEURAL_SUBJECT_SUMMARY_PATH)
neural_subject_df["subject"] = neural_subject_df["subject"].map(normalize_subject_id)

required_cols = {
    "dataset_label",
    "experiment_name",
    "subject",
    "model",
    "model_family",
    "auc_mean",
}

missing = required_cols - set(neural_subject_df.columns)
if missing:
    raise ValueError(f"Neural subject summary is missing columns: {sorted(missing)}")

save_dataframe(
    neural_subject_df,
    TABLE_DIR / "copied_notebook20_subject_level_model_summary.csv",
)

print("Notebook 20 subject summary rows:", len(neural_subject_df))

print("\nRows per model:")
display(
    neural_subject_df
    .groupby(["model", "model_family"])["subject"]
    .count()
    .reset_index(name="n_subject_recordings")
)

print("\nRows per dataset/model:")
display(
    neural_subject_df
    .groupby(["dataset_label", "model"])["subject"]
    .count()
    .unstack()
)

display(round_numeric_columns(neural_subject_df.head(), 4))

In [ ]:
subject_plan_df = (
    neural_subject_df
    [["experiment_name", "dataset_label", "subject"]]
    .drop_duplicates()
    .sort_values(["experiment_name", "subject"])
    .reset_index(drop=True)
)

save_dataframe(
    subject_plan_df,
    TABLE_DIR / "subject_plan_from_notebook20.csv",
)

print("Subject-recordings to evaluate:", len(subject_plan_df))

display(subject_plan_df.groupby(["experiment_name", "dataset_label"])["subject"].count().reset_index(name="n_subjects"))
display(subject_plan_df.head())

In [ ]:
def evaluate_classical_subject_feature_set(
        *,
        exp_name: str,
        subject: str,
        feature_name: str,
        feature_cfg: dict,
):
    cfg = safe_get_experiment(exp_name)

    epoching_cfg = make_epoching_config_from_experiment(cfg)

    subject_data, times, ch_names, sfreq = load_one_subject_grouped(
        dataset=cfg.dataset.dataset,
        task_name=cfg.dataset.task,
        subject=subject,
        epoching_cfg=epoching_cfg,
        chunk_size=cfg.splits.group_chunk_size,
        apply_baseline=True,
        verbose=False,
    )

    X = subject_data.X
    y = subject_data.y
    groups = subject_data.groups

    X_feat, feat_meta = extract_features(
        X=X,
        times=times,
        feature_cfg=feature_cfg,
    )

    folds = build_purged_chunk_splits(
        y=y,
        groups=groups,
        n_splits=getattr(cfg.splits, "n_splits", N_SPLITS_FALLBACK),
        purge_neighbors=getattr(cfg.splits, "purge_neighbor_groups", PURGE_NEIGHBORS_FALLBACK),
    )

    rows = []

    for model_name in CLASSICAL_MODEL_NAMES:
        for fold_id, (tr_idx, te_idx) in enumerate(folds, start=1):
            clf = make_classical_model(model_name)

            X_tr = X_feat[tr_idx]
            X_te = X_feat[te_idx]
            y_tr = y[tr_idx]
            y_te = y[te_idx]

            clf.fit(X_tr, y_tr)

            scores, y_pred = get_scores_and_predictions(clf, X_te)

            row = {
                "experiment_name": exp_name,
                "dataset_label": dataset_label_from_cfg(cfg),
                "subject": normalize_subject_id(subject),
                "feature_set": feature_name,
                "feature_label": feature_cfg["label"],
                "model": f"{model_name}_{feature_name}",
                "base_model": model_name,
                "model_family": "classical",
                "fold": int(fold_id),
                "n_train": int(len(tr_idx)),
                "n_test": int(len(te_idx)),
                "train_target_rate": float(y_tr.mean()),
                "test_target_rate": float(y_te.mean()),
                "n_channels": int(X.shape[1]),
                "n_times_original": int(X.shape[2]),
                "n_features": int(feat_meta["n_features"]),
                "n_bins": feat_meta["n_bins"],
                "feature_time_start": feat_meta["time_start"],
                "feature_time_stop": feat_meta["time_stop"],
            }

            row.update(classification_metric_dict(y_te, scores, y_pred))

            rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
classical_fold_cache = TABLE_DIR / "classical_feature_fold_results.csv"

if classical_fold_cache.exists():
    print("Loading cached classical fold results:", classical_fold_cache)
    classical_fold_df = pd.read_csv(classical_fold_cache)
else:
    all_frames = []

    total_jobs = len(subject_plan_df) * len(FEATURE_CONFIGS)
    job_id = 0

    for _, plan_row in subject_plan_df.iterrows():
        exp_name = plan_row["experiment_name"]
        subject = plan_row["subject"]

        for feature_name, feature_cfg in FEATURE_CONFIGS.items():
            job_id += 1
            print(
                f"[{job_id}/{total_jobs}] "
                f"{exp_name} subject={subject} feature={feature_name}"
            )

            t0 = time.time()

            df_one = evaluate_classical_subject_feature_set(
                exp_name=exp_name,
                subject=subject,
                feature_name=feature_name,
                feature_cfg=feature_cfg,
            )

            df_one["elapsed_sec"] = float(time.time() - t0)

            all_frames.append(df_one)

            partial = pd.concat(all_frames, ignore_index=True)
            save_dataframe(partial, classical_fold_cache)

    classical_fold_df = pd.concat(all_frames, ignore_index=True)
    save_dataframe(classical_fold_df, classical_fold_cache)

print("Classical fold rows:", len(classical_fold_df))

display(round_numeric_columns(classical_fold_df.head(), 4))

print("\nRows per feature/model:")
display(
    classical_fold_df
    .groupby(["feature_set", "base_model"])["subject"]
    .count()
    .unstack()
)

In [ ]:
classical_subject_df = (
    classical_fold_df
    .groupby(
        [
            "dataset_label",
            "experiment_name",
            "subject",
            "feature_set",
            "feature_label",
            "model",
            "base_model",
            "model_family",
        ],
        as_index=False,
    )
    .agg(
        n_folds=("fold", "count"),
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        bal_acc_mean=("bal_acc", "mean"),
        f1_mean=("f1", "mean"),
        n_features=("n_features", "first"),
        n_bins=("n_bins", "first"),
        feature_time_start=("feature_time_start", "first"),
        feature_time_stop=("feature_time_stop", "first"),
        elapsed_sec_sum=("elapsed_sec", "sum"),
    )
    .sort_values(["dataset_label", "subject", "feature_set", "model"])
    .reset_index(drop=True)
)

save_dataframe(
    classical_subject_df,
    TABLE_DIR / "classical_feature_subject_summary.csv",
)

print("Classical subject rows:", len(classical_subject_df))

display(round_numeric_columns(classical_subject_df.head(30), 4))

In [ ]:
# Keep the Notebook 20 neural/classical rows, but mark source clearly.
notebook20_subject_df = neural_subject_df.copy()

notebook20_subject_df["feature_set"] = np.where(
    notebook20_subject_df["model_family"].eq("classical"),
    "raw_flat_notebook20",
    "neural_epoch_input",
)

notebook20_subject_df["feature_label"] = np.where(
    notebook20_subject_df["model_family"].eq("classical"),
    "Raw-flat Notebook 20",
    "Neural epoch input",
)

notebook20_subject_df["base_model"] = notebook20_subject_df["model"]

needed_cols = [
    "dataset_label",
    "experiment_name",
    "subject",
    "feature_set",
    "feature_label",
    "model",
    "base_model",
    "model_family",
    "n_folds",
    "auc_mean",
    "auc_std",
    "pr_auc_mean",
    "bal_acc_mean",
    "f1_mean",
]

notebook20_keep_df = notebook20_subject_df[
    [c for c in needed_cols if c in notebook20_subject_df.columns]
].copy()

save_dataframe(
    notebook20_keep_df,
    TABLE_DIR / "notebook20_subject_summary_prepared_for_comparison.csv",
)

display(round_numeric_columns(notebook20_keep_df.head(20), 4))

In [ ]:
combined_subject_df = pd.concat(
    [
        notebook20_keep_df,
        classical_subject_df[notebook20_keep_df.columns.intersection(classical_subject_df.columns).tolist()],
    ],
    ignore_index=True,
    sort=False,
)

# Remove duplicate raw-flat LDA/LogReg from Notebook 20 if our recomputed raw-flat versions exist.
# Keep the recomputed versions because they are part of the controlled feature-set comparison.
mask_old_raw_classical = (
        combined_subject_df["model_family"].eq("classical")
        & combined_subject_df["feature_set"].eq("raw_flat_notebook20")
)

combined_subject_df = combined_subject_df[~mask_old_raw_classical].copy()

save_dataframe(
    combined_subject_df,
    TABLE_DIR / "combined_neural_and_classical_subject_summary.csv",
)

print("Combined subject rows:", len(combined_subject_df))

print("\nRows per model:")
display(
    combined_subject_df
    .groupby(["model", "model_family"])["subject"]
    .count()
    .reset_index(name="n_subject_recordings")
    .sort_values(["model_family", "model"])
)

display(round_numeric_columns(combined_subject_df.head(), 4))

In [ ]:
overall_summary_df = (
    combined_subject_df
    .groupby(["model", "model_family", "feature_set", "feature_label"], as_index=False)
    .agg(
        n_subject_recordings=("subject", "count"),
        auc_mean=("auc_mean", "mean"),
        auc_std=("auc_mean", "std"),
        auc_median=("auc_mean", "median"),
        pr_auc_mean=("pr_auc_mean", "mean"),
        bal_acc_mean=("bal_acc_mean", "mean"),
        f1_mean=("f1_mean", "mean"),
    )
    .sort_values("auc_mean", ascending=False)
    .reset_index(drop=True)
)

save_dataframe(
    overall_summary_df,
    TABLE_DIR / "overall_model_summary_with_widetimebin.csv",
)

display(round_numeric_columns(overall_summary_df, 4))

In [ ]:
dataset_summary_df = (
    combined_subject_df
    .groupby(["dataset_label", "model", "model_family", "feature_set", "feature_label"], as_index=False)
    .agg(
        n_subject_recordings=("subject", "count"),
        auc_mean=("auc_mean", "mean"),
        auc_std=("auc_mean", "std"),
        auc_median=("auc_mean", "median"),
        pr_auc_mean=("pr_auc_mean", "mean"),
        bal_acc_mean=("bal_acc_mean", "mean"),
        f1_mean=("f1_mean", "mean"),
    )
    .sort_values(["dataset_label", "auc_mean"], ascending=[True, False])
    .reset_index(drop=True)
)

save_dataframe(
    dataset_summary_df,
    TABLE_DIR / "dataset_model_summary_with_widetimebin.csv",
)

display(round_numeric_columns(dataset_summary_df, 4))

In [ ]:
classical_df = combined_subject_df[
    combined_subject_df["model_family"].eq("classical")
].copy()

neural_df = combined_subject_df[
    combined_subject_df["model_family"].eq("neural")
].copy()

best_classical_per_subject_df = (
    classical_df
    .sort_values(
        ["dataset_label", "subject", "auc_mean"],
        ascending=[True, True, False],
    )
    .groupby(["dataset_label", "experiment_name", "subject"])
    .head(1)
    .reset_index(drop=True)
    .rename(columns={
        "model": "best_classical_model",
        "feature_set": "best_classical_feature_set",
        "feature_label": "best_classical_feature_label",
        "auc_mean": "best_classical_auc",
    })
)

best_neural_per_subject_df = (
    neural_df
    .sort_values(
        ["dataset_label", "subject", "auc_mean"],
        ascending=[True, True, False],
    )
    .groupby(["dataset_label", "experiment_name", "subject"])
    .head(1)
    .reset_index(drop=True)
    .rename(columns={
        "model": "best_neural_model",
        "feature_set": "best_neural_feature_set",
        "feature_label": "best_neural_feature_label",
        "auc_mean": "best_neural_auc",
    })
)

best_comparison_df = best_classical_per_subject_df[
    [
        "dataset_label",
        "experiment_name",
        "subject",
        "best_classical_model",
        "best_classical_feature_set",
        "best_classical_feature_label",
        "best_classical_auc",
    ]
].merge(
    best_neural_per_subject_df[
        [
            "dataset_label",
            "experiment_name",
            "subject",
            "best_neural_model",
            "best_neural_feature_set",
            "best_neural_feature_label",
            "best_neural_auc",
        ]
    ],
    on=["dataset_label", "experiment_name", "subject"],
    how="inner",
)

best_comparison_df["best_neural_minus_best_classical"] = (
        best_comparison_df["best_neural_auc"] - best_comparison_df["best_classical_auc"]
)

best_comparison_df["neural_beats_best_classical"] = (
        best_comparison_df["best_neural_minus_best_classical"] > 0
)

save_dataframe(
    best_comparison_df,
    TABLE_DIR / "best_neural_vs_best_classical_per_subject.csv",
)

display(round_numeric_columns(best_comparison_df.head(30), 4))

In [ ]:
best_neural_vs_classical_summary_df = (
    best_comparison_df
    .groupby("dataset_label", as_index=False)
    .agg(
        n_subject_recordings=("subject", "count"),
        mean_best_classical_auc=("best_classical_auc", "mean"),
        mean_best_neural_auc=("best_neural_auc", "mean"),
        mean_neural_minus_classical=("best_neural_minus_best_classical", "mean"),
        median_neural_minus_classical=("best_neural_minus_best_classical", "median"),
        n_neural_beats_classical=("neural_beats_best_classical", "sum"),
        pct_neural_beats_classical=("neural_beats_best_classical", lambda x: 100.0 * float(np.mean(x))),
    )
)

overall_best_row = pd.DataFrame([{
    "dataset_label": "ALL",
    "n_subject_recordings": int(len(best_comparison_df)),
    "mean_best_classical_auc": float(best_comparison_df["best_classical_auc"].mean()),
    "mean_best_neural_auc": float(best_comparison_df["best_neural_auc"].mean()),
    "mean_neural_minus_classical": float(best_comparison_df["best_neural_minus_best_classical"].mean()),
    "median_neural_minus_classical": float(best_comparison_df["best_neural_minus_best_classical"].median()),
    "n_neural_beats_classical": int(best_comparison_df["neural_beats_best_classical"].sum()),
    "pct_neural_beats_classical": 100.0 * float(best_comparison_df["neural_beats_best_classical"].mean()),
}])

best_neural_vs_classical_summary_df = pd.concat(
    [best_neural_vs_classical_summary_df, overall_best_row],
    ignore_index=True,
)

save_dataframe(
    best_neural_vs_classical_summary_df,
    TABLE_DIR / "best_neural_vs_best_classical_summary.csv",
)

display(round_numeric_columns(best_neural_vs_classical_summary_df, 4))

In [ ]:
best_classical_feature_counts_df = (
    best_comparison_df
    .groupby(["dataset_label", "best_classical_model", "best_classical_feature_set"], as_index=False)
    .agg(
        n_subject_recordings=("subject", "count"),
        mean_best_classical_auc=("best_classical_auc", "mean"),
    )
    .sort_values(["dataset_label", "n_subject_recordings"], ascending=[True, False])
)

overall_classical_feature_counts_df = (
    best_comparison_df
    .groupby(["best_classical_model", "best_classical_feature_set"], as_index=False)
    .agg(
        n_subject_recordings=("subject", "count"),
        mean_best_classical_auc=("best_classical_auc", "mean"),
    )
    .sort_values("n_subject_recordings", ascending=False)
)

save_dataframe(
    best_classical_feature_counts_df,
    TABLE_DIR / "best_classical_feature_counts_by_dataset.csv",
)

save_dataframe(
    overall_classical_feature_counts_df,
    TABLE_DIR / "best_classical_feature_counts_overall.csv",
)

print("By dataset:")
display(round_numeric_columns(best_classical_feature_counts_df, 4))

print("\nOverall:")
display(round_numeric_columns(overall_classical_feature_counts_df, 4))

In [ ]:
best_neural_model_counts_df = (
    best_comparison_df
    .groupby(["dataset_label", "best_neural_model"], as_index=False)
    .agg(
        n_subject_recordings=("subject", "count"),
        mean_best_neural_auc=("best_neural_auc", "mean"),
    )
    .sort_values(["dataset_label", "n_subject_recordings"], ascending=[True, False])
)

overall_neural_model_counts_df = (
    best_comparison_df
    .groupby("best_neural_model", as_index=False)
    .agg(
        n_subject_recordings=("subject", "count"),
        mean_best_neural_auc=("best_neural_auc", "mean"),
    )
    .sort_values("n_subject_recordings", ascending=False)
)

save_dataframe(
    best_neural_model_counts_df,
    TABLE_DIR / "best_neural_model_counts_by_dataset.csv",
)

save_dataframe(
    overall_neural_model_counts_df,
    TABLE_DIR / "best_neural_model_counts_overall.csv",
)

print("By dataset:")
display(round_numeric_columns(best_neural_model_counts_df, 4))

print("\nOverall:")
display(round_numeric_columns(overall_neural_model_counts_df, 4))

In [ ]:
shallow_df = neural_df[neural_df["model"].eq("shallowconvnet")].copy()

shallow_vs_classical_df = shallow_df[
    [
        "dataset_label",
        "experiment_name",
        "subject",
        "model",
        "auc_mean",
    ]
].rename(columns={
    "model": "neural_model",
    "auc_mean": "shallowconvnet_auc",
}).merge(
    best_classical_per_subject_df[
        [
            "dataset_label",
            "experiment_name",
            "subject",
            "best_classical_model",
            "best_classical_feature_set",
            "best_classical_auc",
        ]
    ],
    on=["dataset_label", "experiment_name", "subject"],
    how="inner",
)

shallow_vs_classical_df["shallow_minus_best_classical"] = (
        shallow_vs_classical_df["shallowconvnet_auc"] - shallow_vs_classical_df["best_classical_auc"]
)

shallow_vs_classical_df["shallow_beats_best_classical"] = (
        shallow_vs_classical_df["shallow_minus_best_classical"] > 0
)

shallow_summary_df = (
    shallow_vs_classical_df
    .groupby("dataset_label", as_index=False)
    .agg(
        n_subject_recordings=("subject", "count"),
        mean_shallow_auc=("shallowconvnet_auc", "mean"),
        mean_best_classical_auc=("best_classical_auc", "mean"),
        mean_shallow_minus_classical=("shallow_minus_best_classical", "mean"),
        median_shallow_minus_classical=("shallow_minus_best_classical", "median"),
        n_shallow_beats_classical=("shallow_beats_best_classical", "sum"),
        pct_shallow_beats_classical=("shallow_beats_best_classical", lambda x: 100.0 * float(np.mean(x))),
    )
)

shallow_overall_row = pd.DataFrame([{
    "dataset_label": "ALL",
    "n_subject_recordings": int(len(shallow_vs_classical_df)),
    "mean_shallow_auc": float(shallow_vs_classical_df["shallowconvnet_auc"].mean()),
    "mean_best_classical_auc": float(shallow_vs_classical_df["best_classical_auc"].mean()),
    "mean_shallow_minus_classical": float(shallow_vs_classical_df["shallow_minus_best_classical"].mean()),
    "median_shallow_minus_classical": float(shallow_vs_classical_df["shallow_minus_best_classical"].median()),
    "n_shallow_beats_classical": int(shallow_vs_classical_df["shallow_beats_best_classical"].sum()),
    "pct_shallow_beats_classical": 100.0 * float(shallow_vs_classical_df["shallow_beats_best_classical"].mean()),
}])

shallow_summary_df = pd.concat(
    [shallow_summary_df, shallow_overall_row],
    ignore_index=True,
)

save_dataframe(
    shallow_vs_classical_df,
    TABLE_DIR / "shallowconvnet_vs_best_classical_per_subject.csv",
)

save_dataframe(
    shallow_summary_df,
    TABLE_DIR / "shallowconvnet_vs_best_classical_summary.csv",
)

display(round_numeric_columns(shallow_summary_df, 4))

In [ ]:
plot_df = overall_summary_df.sort_values("auc_mean", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))

ax.barh(plot_df["model"], plot_df["auc_mean"])
ax.set_xlabel("Mean subject-level ROC-AUC")
ax.set_title("Overall model comparison: neural vs engineered classical baselines")
ax.grid(axis="x", alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "overall_model_auc_comparison_with_widetimebin.png")
plt.show()

In [ ]:
for dataset_label, g in dataset_summary_df.groupby("dataset_label"):
    plot_df = g.sort_values("auc_mean", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 7))

    ax.barh(plot_df["model"], plot_df["auc_mean"])
    ax.set_xlabel("Mean subject-level ROC-AUC")
    ax.set_title(f"Model comparison with wide-timebin — {dataset_label}")
    ax.grid(axis="x", alpha=0.3)

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"dataset_model_auc_comparison_with_widetimebin_{dataset_label}.png")
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

for dataset_label, g in best_comparison_df.groupby("dataset_label"):
    ax.scatter(
        g["best_classical_auc"],
        g["best_neural_auc"],
        label=dataset_label,
        alpha=0.8,
    )

lim_min = np.nanmin([
    best_comparison_df["best_classical_auc"].min(),
    best_comparison_df["best_neural_auc"].min(),
])
lim_max = np.nanmax([
    best_comparison_df["best_classical_auc"].max(),
    best_comparison_df["best_neural_auc"].max(),
])

ax.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--", linewidth=1)

ax.set_xlabel("Best classical engineered-feature AUC")
ax.set_ylabel("Best neural AUC")
ax.set_title("Best neural model vs best classical baseline")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
save_figure(fig, FIG_DIR / "best_neural_vs_best_classical_scatter.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

for dataset_label, g in shallow_vs_classical_df.groupby("dataset_label"):
    ax.scatter(
        g["best_classical_auc"],
        g["shallowconvnet_auc"],
        label=dataset_label,
        alpha=0.8,
    )

lim_min = np.nanmin([
    shallow_vs_classical_df["best_classical_auc"].min(),
    shallow_vs_classical_df["shallowconvnet_auc"].min(),
])
lim_max = np.nanmax([
    shallow_vs_classical_df["best_classical_auc"].max(),
    shallow_vs_classical_df["shallowconvnet_auc"].max(),
])

ax.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--", linewidth=1)

ax.set_xlabel("Best classical engineered-feature AUC")
ax.set_ylabel("ShallowConvNet AUC")
ax.set_title("ShallowConvNet vs best classical baseline")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
save_figure(fig, FIG_DIR / "shallowconvnet_vs_best_classical_scatter.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    best_comparison_df["best_neural_minus_best_classical"],
    bins=15,
    alpha=0.8,
)

ax.axvline(0.0, linewidth=1)
ax.set_xlabel("Best neural AUC − best classical AUC")
ax.set_ylabel("Number of subject-recordings")
ax.set_title("Subject-level advantage: best neural vs best classical")
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "best_neural_minus_best_classical_histogram.png")
plt.show()

In [ ]:
print("=" * 90)
print("Wide-timebin vs neural benchmark interpretation helper")
print("=" * 90)

print("\nOverall model ranking:")
display(round_numeric_columns(overall_summary_df, 4))

print("\nBest neural vs best classical summary:")
display(round_numeric_columns(best_neural_vs_classical_summary_df, 4))

print("\nShallowConvNet vs best classical summary:")
display(round_numeric_columns(shallow_summary_df, 4))

print("\nBest classical feature counts overall:")
display(round_numeric_columns(overall_classical_feature_counts_df, 4))

print("\nBest neural model counts overall:")
display(round_numeric_columns(overall_neural_model_counts_df, 4))

print("\nUse this decision rule:")
print("- If ShallowConvNet or another neural model still beats best classical overall, say neural models can outperform classical only with the right architecture.")
print("- If wide-timebin LDA/LogReg beats ShallowConvNet, say engineered classical features remain stronger than the tested neural models.")
print("- If the difference is tiny, say the result is competitive rather than decisive.")

In [ ]:
print("Outputs saved under:", OUT_ROOT)

print("\nTables:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print(" -", p)

print("\nFigures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" -", p)